In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date","2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

movies_df = spark.read.table("movie_silver.movies") \
                        .filter(f"file_date = '{v_file_date}'")
countries_df = spark.read.table("movie_silver.countries")
movies_countries_df = spark.read.table("movie_silver.productions_countries") \
                        .filter(f"file_date = '{v_file_date}'")  

In [0]:
window_spec = (
    Window
    .partitionBy("year_release_date")
    .orderBy(
        F.desc("total_budget"),
        F.desc("total_revenue")
    )
)

In [0]:
result_df = (
    movies_df.alias("m")
    .filter(F.col("year_release_date") >= 2015)
    .join(
        movies_countries_df.alias("mc"),
        F.col("m.movie_id") == F.col("mc.movie_id"),
        "inner"
    )
    .join(
        countries_df.alias("c"),
        F.col("mc.country_id") == F.col("c.country_id"),
        "inner"
    )
    .groupBy(
        "year_release_date",
        "country_name"
    )
    .agg(
        F.sum("budget").alias("total_budget"),
        F.sum("revenue").alias("total_revenue")
    )
    .withColumn(
        "dense_rank",
        F.dense_rank().over(window_spec)
    )
    .withColumn("created_date",F.lit(v_file_date))
)

In [0]:
merge_condition = "tgt.year_release_date = src.year_release_date AND tgt.country_name = src.country_name AND tgt.created_date = src.created_date"
merge_delta_lake(result_df,"movie_gold","results_group_movie_country",merge_condition,"created_date")

In [0]:
%sql
SELECT created_date,
COUNT(1) 
FROM movie_gold.results_group_movie_country
GROUP BY created_date

created_date,count(1)
2024-12-23,2
2024-12-30,45
